In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters: ", len(text))

print(text[:100])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [2]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [3]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]             # 將字母編碼成數字
decode = lambda l: ''.join([itos[i] for i in l])    # 將數字解碼回字母

print(encode("hii there"))                          # 示範編碼結果，數字應該介於0~64
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [4]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)         # 使用長整數資料型態

print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [5]:
block_size = 8

n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [6]:
torch.manual_seed(1024)
batch_size = 4                  # how many independent sequences will we process in parallel?
block_size = 8                  # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    idx = torch.randint(len(data) - block_size, (batch_size,))      # 隨機生成index，最小值預設為0，最大值因為輸出最多只到len(data)，故往前扣除block_size

    x = torch.stack([data[i:i+block_size] for i in idx])            # data張量中，從idx開始，到idx+block_size，中的所有張量值
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])        # data張量中，從idx+1開始，到idx+block_size+1，中的所有張量值
    
    return x, y                                                     # 因為此次訓練是包含1~8個輸入，來預測輸出，因此y不是從idx+8開始

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):                     # batch dimension
    for t in range(block_size):                 # time dimension
        
        context = xb[b, :t+1]                   # 表示xb在該批次中，從0到t的張量值
        target = yb[b,t]                        # 表示yb在該批次中，第t位的張量值，這基本會等於xb的t+1位

        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[58, 46, 39, 58,  1, 57, 53,  1],
        [12,  1,  1, 51, 63,  1, 52, 47],
        [46, 44, 59, 50,  1, 57, 53, 59],
        [43, 50, 57, 43,  0, 32, 53,  1]])
targets:
torch.Size([4, 8])
tensor([[46, 39, 58,  1, 57, 53,  1, 50],
        [ 1,  1, 51, 63,  1, 52, 47, 43],
        [44, 59, 50,  1, 57, 53, 59, 50],
        [50, 57, 43,  0, 32, 53,  1, 46]])
----
when input is [58] the target: 46
when input is [58, 46] the target: 39
when input is [58, 46, 39] the target: 58
when input is [58, 46, 39, 58] the target: 1
when input is [58, 46, 39, 58, 1] the target: 57
when input is [58, 46, 39, 58, 1, 57] the target: 53
when input is [58, 46, 39, 58, 1, 57, 53] the target: 1
when input is [58, 46, 39, 58, 1, 57, 53, 1] the target: 50
when input is [12] the target: 1
when input is [12, 1] the target: 1
when input is [12, 1, 1] the target: 51
when input is [12, 1, 1, 51] the target: 63
when input is [12, 1, 1, 51, 63] the target: 1
when input is [12, 1, 1, 

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)           # 形成一個65 X 65的矩陣

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)
        
        if (targets==None):
            loss=None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # print(logits.shape)
            # print(targets.shape)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):                        # 用來生成，其實就是inference
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            
            logits = logits[:, -1, :]
            
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [14]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(xb.shape)
print(yb.shape)
print(logits.shape)
print(logits)
print(loss)

torch.Size([32, 8])
torch.Size([32, 8])
torch.Size([256, 65])
tensor([[ 0.3978, -0.8861,  0.5047,  ...,  0.3131, -0.9071, -1.2439],
        [-0.0147, -1.0378,  0.3711,  ..., -1.5345, -0.7909,  1.0203],
        [-0.1488, -0.6025,  0.5220,  ...,  1.8071, -2.3914, -2.0555],
        ...,
        [-0.3156,  0.3245,  0.1243,  ...,  0.0432, -1.2561,  2.9174],
        [-0.1488, -0.6025,  0.5220,  ...,  1.8071, -2.3914, -2.0555],
        [ 0.5874, -0.4772,  1.5582,  ..., -0.5387, -1.6816, -1.1471]],
       grad_fn=<ViewBackward0>)
tensor(4.6530, grad_fn=<NllLossBackward0>)


In [15]:
idx = torch.zeros((1, 1), dtype=torch.long)

print(decode(m.generate(idx = idx, max_new_tokens=100)[0].tolist()))


aUZlbVicX.wsDJKYoobKNyEjLoSV-;AAU Pw3kcopKE3cClMVVkeNGt!hum;Ir: pY'Y&,
k!seuHk:uoo,tBcjcImRu.pPBw
YQ


In [16]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [24]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)                        # 計算預測值與loss

    optimizer.zero_grad(set_to_none=True)           # 將梯度歸零

    loss.backward()                                 # 計算梯度

    optimizer.step()                                # 更新參數

print(loss.item())

2.4316749572753906


In [26]:
idx = torch.zeros((1, 1), dtype=torch.long)

print(decode(m.generate(idx = idx, max_new_tokens=1000)[0].tolist()))


Wie n se isusy den?

D:
A:
Bu my as ame s ab.
Bindes tothaisersispe hef doatorit friven ICH:
BENRoork,
A bolo?
FO: l Fimbed!
Touad o s eney
AUpll s herds?
Dusum ind s ak LLetos mit Ed med?
Fre t dre, nd g soreeisstt.
Ththe su owey, hall h;-d:
Foucknd assas?-lthiter d,Thand nts, noupre k lor angisave,

Th
I:
Sed I h$powoun, KENCICIDI cy, the, he, ndit pathern thinds t ald u fond, paithendorcen, co tat s?
Whe thathe g hene t, l o ance faru fowm hist fon

That p'ld tacerivomowe,
Whafordstl.
Haknbat y sl beteanoutherve wei-m in tt AUCIfo rare pr bove tl m tin buthond th, bees foro 'GRUDedy,
orenaubr thas , tou g'd i,
MI brThal I aukeet ier


UD: brds ofove po nderu ll arce shn lllveeas w fe denoss gr o I ake bll ando KANo,
Bu th.
Sil thanens helsponofuserat ar tho,
A jmp
Thesy otof d be ROFRUSI:
CEN ber f ain mbesharoralin.
St 't mig ater a lethye wde wasthod f mb.
WA:

ad shalYouers! brema'su.
ANSjlas II wit sirofo aurugy I's heleveind.
Walourdlleicu to athief thed blesst' pr,
CHENo O: l